In [0]:
%python
# Limpieza de widgets previos al inicio del rollback para evitar conflictos
dbutils.widgets.removeAll()

In [0]:
%python
# Parámetros de entrada: nombre del storage account y catálogo a limpiar
dbutils.widgets.text("storageLocation","adlsftrsmardatadev01")
dbutils.widgets.text("catalogo","cat_ftr_smartdata_dev")

In [0]:
%python
# Recuperación de parámetros y construcción de las rutas base por capa en ADLS Gen2
storageLocation = dbutils.widgets.get("storageLocation")
catalogo = dbutils.widgets.get("catalogo")

rutaBronze = f"abfss://bronze@{storageLocation}.dfs.core.windows.net"
rutaSilver = f"abfss://silver@{storageLocation}.dfs.core.windows.net"
rutaGolden = f"abfss://gold@{storageLocation}.dfs.core.windows.net"

### Eliminación de tablas Bronze
Elimina los registros en Unity Catalog y los archivos físicos en ADLS para `bronze.ventas` y `bronze.productos`.

In [0]:
-- Eliminación de tablas Delta Bronze del catálogo Unity Catalog
-- DROP TABLE solo elimina el registro; el paso siguiente elimina los archivos físicos en ADLS
DROP TABLE IF EXISTS `${catalogo}`.bronze.ventas;
DROP TABLE IF EXISTS `${catalogo}`.bronze.productos;

In [0]:
%python
# Eliminación de los archivos Delta en ADLS para las tablas Bronze
# Se ejecuta después del DROP TABLE para limpiar también el almacenamiento físico
dbutils.fs.rm(f"{rutaBronze}/ventas", True)
dbutils.fs.rm(f"{rutaBronze}/productos", True)

### Eliminación de tablas Silver
Elimina el registro en Unity Catalog y los archivos físicos en ADLS para `silver.ventas_productos_categorias`.

In [0]:
-- Eliminación de la tabla Delta Silver del catálogo Unity Catalog
DROP TABLE IF EXISTS ${catalogo}.silver.ventas_productos_categorias;


In [0]:
%python
# Eliminación de los archivos Delta en ADLS para la tabla Silver
dbutils.fs.rm(f"{rutaSilver}/ventas_productos_categorias", True)

### Eliminación de tablas Gold
Elimina los registros en Unity Catalog y los archivos físicos en ADLS para las 3 tablas Gold:
`golden.ventas_diarias_tienda`, `golden.ventas_categoria_mes` y `golden.kpi_vendedor_mes`.

In [0]:
-- Eliminación de las 3 tablas Delta Gold del catálogo Unity Catalog
-- Incluye la tabla kpi_vendedor_mes añadida al pipeline en la última actualización
DROP TABLE IF EXISTS ${catalogo}.golden.ventas_diarias_tienda;
DROP TABLE IF EXISTS ${catalogo}.golden.ventas_categoria_mes;
DROP TABLE IF EXISTS ${catalogo}.golden.kpi_vendedor_mes;

In [0]:
%python
# Eliminación de los archivos Delta en ADLS para las 3 tablas Gold
dbutils.fs.rm(f"{rutaGolden}/ventas_diarias_tienda", True)
dbutils.fs.rm(f"{rutaGolden}/ventas_categoria_mes", True)
dbutils.fs.rm(f"{rutaGolden}/kpi_vendedor_mes", True)